# Validate and plan an experiment

This tutorial creates a minimal experiment configuration, validates it, and materializes a dry-run plan. It does not start model training. The resulting plan makes parameter-grid expansion, deterministic seeds, and output paths visible before compute resources are allocated.

## Locate the repository

Run the notebook from any directory inside the repository. The following cell locates `pyproject.toml` without depending on the notebook's nested documentation path.

In [ ]:
from pathlib import Path

repository_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
)
tutorial_output = repository_root / "data" / "tutorial_workspace" / "executions"
tutorial_output.mkdir(parents=True, exist_ok=True)
repository_root

## Write a training-only experiment configuration

An experiment names an importable task entrypoint, supplies its base parameters, and optionally expands selected parameters as a Cartesian grid. The placeholder entrypoint below is suitable for inspecting configuration and planning. Replace it with the project training entrypoint before executing `run`.

In [ ]:
experiment_yaml = tutorial_output / "tutorial-experiment.yaml"
experiment_yaml.write_text(
    """schema_version: 1
experiment:
  name: tutorial-training
task:
  entrypoint: package.module:run
  parameters:
    workspace: data/tutorial_workspace
    training:
      batch_size: 32
runs:
  repetitions: 2
  base_seed: 1912
grid:
  training.batch_size: [16, 32]
execution:
  backend: local
  max_parallel_runs: 1
  work_directory: data/tutorial_workspace/executions/tutorial-training
reports: []
""",
    encoding="utf-8",
)
print(experiment_yaml.read_text(encoding="utf-8"))

## Validate and materialize the plan

`validate` checks the schema and configured entrypoint. Because this example deliberately uses a placeholder entrypoint, inspect the schema with the Python loader first. After replacing the entrypoint with a real training function, use the commented CLI commands to perform the full preflight and dry run.

In [ ]:
from msi_autoencoder_wrapper.execution.configuration import load_experiment_config

configuration = load_experiment_config(experiment_yaml)
print(configuration.experiment.name)
print(configuration.runs.repetitions, configuration.grid)

# After setting a real task.entrypoint:
# !msi-wrapper validate {experiment_yaml}
# !msi-wrapper run {experiment_yaml} --dry-run

## Continue with the full reference

The [experiment configuration guide](../../how-to/experiment-execution/experiment-configuration.md) documents every top-level field. The [CLI guide](../../how-to/experiment-execution/experiment-cli.md) covers validation, planning, local execution, Slurm submission, and report rendering.